# Mejorando el pronóstico de demanda

Continuación de `01_pronostico_demanda.ipynb`. Allí una sola prueba (8 semanas) mostró una mejora de 4,6 % sobre la línea base. Aquí preguntamos:

1. ¿Esa mejora se sostiene si probamos en **varias ventanas** del tiempo (validación *walk-forward*)?
2. ¿Ayudan los **días festivos de México** y las **promociones** como variables?
3. ¿Le gana Holt-Winters a SARIMA o a un modelo de árboles?
4. ¿Cómo damos un **rango** ("entre X y Y") en vez de un solo número?

> Usa **datos sintéticos** (`scripts/generate_sample_data_eventos.py`) en los que el efecto de festivos (+25 %) y promociones (+30 %) está incluido a propósito. Eso sirve para practicar la técnica; no demuestra que en un negocio real ayuden. En datos reales se mide igual y el resultado puede ser otro.

## 0. Parámetros

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/ventas_ejemplo_eventos.csv")   # columnas: fecha, ventas, promo
SEASON = 7        # estacionalidad semanal
HORIZON = 14      # días a pronosticar en cada ventana
N_VENTANAS = 8    # cuántas ventanas de prueba
PASO = 28         # días que se recorre el origen entre ventanas
N_RANGO = 20      # ventanas (cada 14 días) para calibrar y validar el rango
N_CALIB = 14      # las más antiguas calibran el rango; las demás lo validan

In [ ]:
import warnings

import holidays
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")
pd.options.display.float_format = "{:,.2f}".format

## 1. Datos

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro {DATA_PATH}. Genéralo con:  python scripts/generate_sample_data_eventos.py"
    )

df = (
    pd.read_csv(DATA_PATH, parse_dates=["fecha"])
      .drop_duplicates("fecha").set_index("fecha").sort_index().asfreq("D")
)
assert df["ventas"].isna().sum() == 0, "Hay huecos: límpialos como en el notebook 01 antes de seguir"
serie = df["ventas"]
promo = df["promo"].fillna(0).astype(int)
print(serie.index.min().date(), "→", serie.index.max().date(), f"({len(serie)} días, {promo.sum()} días de promoción)")

## 2. Variables de calendario y eventos

El modelo de árboles no "ve" el tiempo: hay que describírselo con variables. Las que se conocen **de antemano** (día de la semana, época del año, festivos, promociones planeadas) pueden usarse para pronosticar el futuro sin trampa.

In [ ]:
FESTIVOS = holidays.country_holidays("MX", years=range(serie.index.year.min(), serie.index.year.max() + 2))

def calendario(idx, con_eventos):
    X = pd.DataFrame(index=idx)
    X["dia_semana"] = idx.day_of_week
    X["anio_sen"] = np.sin(2 * np.pi * idx.day_of_year / 365.25)
    X["anio_cos"] = np.cos(2 * np.pi * idx.day_of_year / 365.25)
    X["t"] = (idx - serie.index[0]).days
    if con_eventos:
        X["festivo"] = [int(d.date() in FESTIVOS) for d in idx]
        X["promo"] = promo.reindex(idx).fillna(0).astype(int).to_numpy()
    return X

calendario(serie.index[:3], con_eventos=True)

## 3. Los modelos

Cada función recibe el pasado (`train`) y devuelve el pronóstico de los siguientes `h` días:

- **Línea base:** el mismo día de la semana pasada.
- **Holt-Winters:** tendencia amortiguada + estacionalidad semanal (el del notebook 01).
- **SARIMA:** modelo clásico de series de tiempo con parte estacional semanal.
- **Árboles sin eventos:** `HistGradientBoostingRegressor` solo con calendario.
- **Árboles con eventos:** lo anterior más festivos y promociones.

In [ ]:
def pron_base(train, h):
    return np.tile(train.iloc[-SEASON:].to_numpy(), int(np.ceil(h / SEASON)))[:h]

def pron_holt_winters(train, h):
    m = ExponentialSmoothing(train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=SEASON).fit()
    return m.forecast(h).to_numpy()

def pron_sarima(train, h):
    m = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, SEASON)).fit(disp=False)
    return m.forecast(h).to_numpy()

def pron_arboles(train, h, con_eventos):
    futuro = pd.date_range(train.index[-1] + pd.Timedelta(days=1), periods=h, freq="D")
    m = HistGradientBoostingRegressor(
        categorical_features=["dia_semana"], learning_rate=0.05, max_iter=300, random_state=0
    ).fit(calendario(train.index, con_eventos), train)
    return m.predict(calendario(futuro, con_eventos))

MODELOS = {
    "Línea base": pron_base,
    "Holt-Winters": pron_holt_winters,
    "SARIMA": pron_sarima,
    "Árboles sin eventos": lambda tr, h: pron_arboles(tr, h, False),
    "Árboles con eventos": lambda tr, h: pron_arboles(tr, h, True),
}

## 4. Validación walk-forward

En el notebook 01 se probó una sola vez. Aquí se repite **8 veces**: en cada ventana se entrena solo con lo anterior a ese punto y se pronostican los siguientes 14 días; luego el origen avanza 28 días. Así sabemos si un modelo es consistentemente mejor o si tuvo suerte en una ventana.

In [ ]:
cortes = [len(serie) - HORIZON - PASO * k for k in range(N_VENTANAS - 1, -1, -1)]

reales = np.array([serie.iloc[c:c + HORIZON].to_numpy() for c in cortes])
pronosticos = {nombre: [] for nombre in MODELOS}
for c in cortes:
    train = serie.iloc[:c]
    for nombre, f in MODELOS.items():
        pronosticos[nombre].append(f(train, HORIZON))
pronosticos = {n: np.array(p) for n, p in pronosticos.items()}

print("Ventanas de prueba (inicio de cada pronóstico):")
print([serie.index[c].strftime("%Y-%m-%d") for c in cortes])

## 5. Resultados

- **MAE:** error absoluto medio (en unidades de venta). **WAPE:** error total entre ventas totales (%).
- **Peor ventana:** el MAE de la ventana donde le fue peor. Un buen modelo no solo es bueno en promedio, también es estable.
- **Ventanas ganadas:** en cuántas de las 8 ventanas tuvo el menor MAE.

In [ ]:
mae_ventana = {n: np.abs(reales - p).mean(axis=1) for n, p in pronosticos.items()}
ganadoras = pd.Series(
    [min(mae_ventana, key=lambda n: mae_ventana[n][i]) for i in range(N_VENTANAS)]
).value_counts()

tabla = pd.DataFrame({
    n: {
        "MAE": np.abs(reales - p).mean(),
        "WAPE %": 100 * np.abs(reales - p).sum() / reales.sum(),
        "Peor ventana (MAE)": mae_ventana[n].max(),
        "Ventanas ganadas": int(ganadoras.get(n, 0)),
    }
    for n, p in pronosticos.items()
}).T.sort_values("MAE")
tabla["Mejora vs. línea base"] = 1 - tabla["MAE"] / tabla.loc["Línea base", "MAE"]
display(tabla.style.format({"MAE": "{:.2f}", "WAPE %": "{:.2f}", "Peor ventana (MAE)": "{:.2f}", "Mejora vs. línea base": "{:.1%}"}))

In [ ]:
fig = go.Figure()
for n, v in mae_ventana.items():
    fig.add_trace(go.Scatter(x=[serie.index[c].strftime("%Y-%m-%d") for c in cortes], y=v, name=n, mode="lines+markers"))
fig.update_layout(title="MAE por ventana de prueba", template="plotly_white", height=420, legend=dict(orientation="h"), yaxis_title="MAE")
fig.show()

## 6. ¿Qué aportan los festivos y las promociones?

Comparación directa de los dos modelos de árboles: la **única** diferencia entre ellos son las variables de eventos.

In [ ]:
sin, con = tabla.loc["Árboles sin eventos", "MAE"], tabla.loc["Árboles con eventos", "MAE"]
print(f"MAE sin eventos: {sin:.2f}  |  con eventos: {con:.2f}  |  reducción del error: {1 - con / sin:.1%}")

mascara = np.array([[(promo.get(serie.index[c + j], 0) == 1) or (serie.index[c + j].date() in FESTIVOS)
                     for j in range(HORIZON)] for c in cortes])
for n in ["Árboles sin eventos", "Árboles con eventos"]:
    err = np.abs(reales - pronosticos[n])
    print(f"{n:22s} MAE en días de evento: {err[mascara].mean():6.2f}  |  en días normales: {err[~mascara].mean():6.2f}")

## 7. Un rango en vez de un número

Un pronóstico de "175" engaña: lo honesto es "entre 160 y 190". Repetimos el pronóstico del mejor modelo en **20 ventanas** (más que en la tabla, porque un rango necesita muchos errores para estimarse bien). Con los errores de las 14 ventanas más antiguas calibramos un rango del 80 % y comprobamos si cubre de verdad el 80 % de los valores en las 6 más recientes, que no se usaron para calibrar.

In [ ]:
mejor = tabla.index[0]
cortes_rango = [len(serie) - HORIZON - 14 * k for k in range(N_RANGO - 1, -1, -1)]
err_rango = np.array([
    serie.iloc[c:c + HORIZON].to_numpy() - MODELOS[mejor](serie.iloc[:c], HORIZON) for c in cortes_rango
])                                                   # real - pronóstico, forma (ventanas, días)

margen = np.quantile(np.abs(err_rango[:N_CALIB]), 0.80)
dentro = (np.abs(err_rango[N_CALIB:]) <= margen).mean()
print(f"Mejor modelo: {mejor}")
print(f"Rango 80 %: pronóstico ± {margen:.1f}")
print(f"Cobertura en las ventanas de validación: {dentro:.0%} (objetivo: 80 %)")

## 8. Pronóstico final

Se reentrena el mejor modelo con **toda** la historia. Se asume que **no hay promociones** en los próximos días (cámbialo si tienes alguna planeada); los festivos sí se toman del calendario.

In [ ]:
futuro_idx = pd.date_range(serie.index[-1] + pd.Timedelta(days=1), periods=HORIZON, freq="D")
pron = pd.Series(MODELOS[mejor](serie, HORIZON), index=futuro_idx)
final = pd.DataFrame({
    "pronostico": pron,
    "minimo": pron - margen,
    "maximo": pron + margen,
}).round(1)
final["dia"] = ["Lun Mar Mié Jue Vie Sáb Dom".split()[d] for d in final.index.day_of_week]
final["festivo"] = [FESTIVOS.get(d.date(), "") for d in final.index]
final

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index[-90:], y=serie.iloc[-90:], name="Histórico", line=dict(color="#9AA5B8")))
fig.add_trace(go.Scatter(x=final.index, y=final["maximo"], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=final.index, y=final["minimo"], fill="tonexty", fillcolor="rgba(14,165,233,.18)",
                         line=dict(width=0), name="Rango 80 %", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=final.index, y=final["pronostico"], name=f"Pronóstico ({mejor})", line=dict(color="#0EA5E9", width=3)))
fig.update_layout(title=f"Pronóstico de ventas: próximos {HORIZON} días, con rango", template="plotly_white", height=440, legend=dict(orientation="h"))
fig.show()

REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)
fig.write_html(REPORTS / "forecast_mejorado.html", include_plotlyjs="cdn", full_html=True)
final.to_csv(REPORTS / "pronostico_mejorado.csv", index_label="fecha")

## 9. Qué aprendimos *(completar con tus datos reales)*

- **Consistencia:** [¿el modelo ganador lo es en la mayoría de las ventanas o solo en promedio?]
- **Eventos:** [cuánto bajó el error al agregar festivos y promociones, y en qué tipo de día]
- **Rango:** [¿la cobertura se acercó al 80 %?; si quedó por debajo, el rango es demasiado optimista]
- **Límites:** [datos sintéticos, 3 años de historia, sin clima, sin precios, promociones asumidas como conocidas de antemano]
- **Siguientes pasos:** [agregar clima o precio, probar otros modelos, medir el impacto en merma o inventario]